In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Subtract, Multiply, concatenate, Dot
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# Kaggle  : set USE_AUGMENTED = True/False sesuai kebutuhan
# Lokal   : ubah DATA_DIR ke path lokal

# DATA_DIR      = "/kaggle/input/datasets/harimurtiadi/indobert-embedding"        # IndoBERT
DATA_DIR      = "/kaggle/input/siamese-data"                                    # 17 IDPSJ
# DATA_DIR      = "/kaggle/input/datasets/harimurtiadi/embedding-fasttext-coba"   # Asli
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True            # True  → pakai aug_*.npy + aug_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'final_' if USE_AUGMENTED else ''
meta_f = 'final_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

TensorFlow  : 2.19.0
GPU tersedia: 1
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  final_questions_emb.npy        -> OK
  final_answerkeys_emb.npy       -> OK
  final_answers_emb.npy          -> OK
  final_metadata.pkl             -> OK


In [3]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

=== Hasil Load ===
answers_emb    : (1743, 80, 300)    (per sampel)
uniq_q_emb     : (12, 135, 300)   (kompak, per IDPSJ)
questions_emb  : (1743, 135, 300)  (rekonstruksi)
answerkeys_emb : (1743, 90, 300) (rekonstruksi)

Metadata       : 1743 rows
Kolom metadata : ['IDJwb', 'IDPSJ', 'grade', 'psj_idx']

IDPSJ unik     : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

Distribusi grade:
grade
1     175
2     177
3     163
4     161
5     157
6     167
7     169
8     170
9     163
10    241
Name: count, dtype: int64


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# KONFIGURASI
# ══════════════════════════════════════════════════════════════════════════════

BILSTM_UNITS = 48          # Kecil → kurangi parameter ~70%, kurangi overfitting
DROPOUT      = 0.4
EPOCHS       = 150
BATCH_SIZE   = 16          # Lebih kecil → gradient update lebih sering
PATIENCE     = 20          # Lebih tinggi → beri kesempatan model kecil converge
LR_INIT      = 5e-4
L2_REG       = 1e-4
GRAD_CLIP    = 1.0

# PERINGATAN: recurrent_dropout > 0 mematikan cuDNN kernel → training jauh lebih lambat.
# Set REC_DROPOUT = 0.0 untuk kecepatan GPU penuh; 0.2 untuk regularisasi ekstra.
REC_DROPOUT  = 0.0         # Ganti ke 0.2 jika mau recurrent dropout (non-cuDNN)


# ══════════════════════════════════════════════════════════════════════════════
# MODEL v1: Lightweight — hanya fitur relasi (tanpa representasi individual)
# ══════════════════════════════════════════════════════════════════════════════

def build_model_v1(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                   bilstm_units=BILSTM_UNITS, dropout=DROPOUT):
    """
    Lightweight Siamese — hanya fitur RELASI di merged, bukan representasi individual.

    Filosofi: dengan tidak memasukkan eq, ea, eak secara individual, model tidak
    bisa "menghapal" pola per-prompt → lebih generalizable untuk LOPO.

    Merged: abs_diff(96) + had_prod(96) + cos_ak_a(1) + cos_q_a(1) = 194D
    Head  : Dense(32, L2) → Dense(1)
    ~60K parameter (vs ~300K+ sebelumnya)
    """
    bilstm_q = Bidirectional(
        LSTM(bilstm_units, return_sequences=False, recurrent_dropout=REC_DROPOUT),
        name='bilstm_question'
    )
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=False, recurrent_dropout=REC_DROPOUT),
        name='shared_bilstm'
    )

    inp_q  = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a  = Input(shape=(a_seq_len,  emb_dim), name='inp_a')

    eq  = bilstm_q(inp_q)
    eak = shared_bilstm(inp_ak)
    ea  = shared_bilstm(inp_a)

    abs_diff     = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_ak_a')([eak, ea])
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_q_a')([eq, ea])

    # Hanya fitur relasi — tidak ada eq/ea/eak individual
    merged = concatenate([abs_diff, had_prod, cos_sim_ak_a, cos_sim_q_a], name='merged')

    x   = Dense(32, activation='relu', kernel_regularizer=l2(L2_REG))(merged)
    x   = Dropout(dropout)(x)
    out = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out,
                  name='siamese_bilstm_lopo_v1')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_INIT, clipnorm=GRAD_CLIP),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    return model


# ══════════════════════════════════════════════════════════════════════════════
# MODEL v2: + Attention Pooling (alternatif — jalankan jika v1 kurang baik)
# ══════════════════════════════════════════════════════════════════════════════

def _attention_pool(lstm_out, name_prefix):
    score   = Dense(1, activation='tanh', name=f'{name_prefix}_score')(lstm_out)
    flat    = tf.keras.layers.Flatten(name=f'{name_prefix}_flat')(score)
    weight  = tf.keras.layers.Activation('softmax', name=f'{name_prefix}_weight')(flat)
    w_exp   = Lambda(lambda x: tf.expand_dims(x, -1), name=f'{name_prefix}_expand')(weight)
    ctx     = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=1),
                     name=f'{name_prefix}_ctx')([lstm_out, w_exp])
    return ctx


def build_model_v2(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                   bilstm_units=BILSTM_UNITS, dropout=DROPOUT):
    """
    Sama seperti v1 tapi BiLSTM return_sequences=True + attention pooling.
    Lebih baik jika jawaban siswa panjang dan perlu fokus ke kata kunci.
    """
    bilstm_q = Bidirectional(
        LSTM(bilstm_units, return_sequences=True, recurrent_dropout=REC_DROPOUT),
        name='bilstm_question'
    )
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=True, recurrent_dropout=REC_DROPOUT),
        name='shared_bilstm'
    )

    inp_q  = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a  = Input(shape=(a_seq_len,  emb_dim), name='inp_a')

    eq  = _attention_pool(bilstm_q(inp_q),          name_prefix='q')
    eak = _attention_pool(shared_bilstm(inp_ak),     name_prefix='ak')
    ea  = _attention_pool(shared_bilstm(inp_a),      name_prefix='a')

    abs_diff     = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_ak_a')([eak, ea])
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_q_a')([eq, ea])

    merged = concatenate([abs_diff, had_prod, cos_sim_ak_a, cos_sim_q_a], name='merged')

    x   = Dense(32, activation='relu', kernel_regularizer=l2(L2_REG))(merged)
    x   = Dropout(dropout)(x)
    out = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out,
                  name='siamese_bilstm_lopo_v2_attn')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_INIT, clipnorm=GRAD_CLIP),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    return model


# ── Pilih model yang dipakai ──────────────────────────────────────────────────
build_model = build_model_v1   # Ganti ke build_model_v2 untuk versi attention


# ── Preview ───────────────────────────────────────────────────────────────────
_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2]
)
_tmp.summary()
del _tmp

In [ ]:
# ── LOPO Cross-Validation (Leave-One-Prompt-Out) ──────────────────────────────
# Data augmentasi HANYA digunakan pada split train.
# Val & Test selalu menggunakan data ASLI (is_synthetic == False).

idpsj_list = sorted(metadata['IDPSJ'].unique())
n_parts    = len(idpsj_list)
y_all      = metadata['grade'].values.astype(np.float32)

if 'is_synthetic' in metadata.columns:
    is_real = ~metadata['is_synthetic'].values
else:
    is_real = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

print(f"Total data    : {len(metadata)}")
print(f"Data asli     : {is_real.sum()}")
print(f"Data sintetis : {(~is_real).sum()}")

fold_results = []

for i, test_id in enumerate(idpsj_list):
    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test={test_id}  |  Val={val_id}")

    train_mask = metadata['IDPSJ'].isin(train_ids)
    train_idx  = metadata.index[train_mask].values
    val_idx    = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    test_idx   = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_tr   = get_split(questions_emb,  train_idx)
    X_ak_tr  = get_split(answerkeys_emb, train_idx)
    X_a_tr   = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te   = get_split(questions_emb,  test_idx)
    X_ak_te  = get_split(answerkeys_emb, test_idx)
    X_a_te   = get_split(answers_emb,    test_idx)

    model = build_model(
        q_seq_len    = questions_emb.shape[1],
        ak_seq_len   = answerkeys_emb.shape[1],
        a_seq_len    = answers_emb.shape[1],
        emb_dim      = answers_emb.shape[2],
        bilstm_units = BILSTM_UNITS,
        dropout      = DROPOUT
    )

    grade_int          = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map           = dict(zip(unique_g, counts_g))
    n_kelas            = len(unique_g)
    raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g])
                                   for g in grade_int])
    sample_w           = raw_w / raw_w.mean()
    print(f"  Sample weight  min={sample_w.min():.2f}  max={sample_w.max():.2f}  "
          f"mean={sample_w.mean():.2f}")

    model_path = os.path.join(OUT_DIR, f'model_fold_{i+1:02d}.keras')
    callbacks  = [
        EarlyStopping(
            monitor='val_mae', patience=PATIENCE,
            restore_best_weights=True, mode='min', verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_mae', factor=0.5, patience=7,
            min_lr=1e-6, mode='min', verbose=1
        ),
        ModelCheckpoint(
            model_path, monitor='val_mae',
            save_best_only=True, mode='min', verbose=0
        ),
    ]

    print("  Memulai proses training...")
    model.fit(
        [X_q_tr, X_ak_tr, X_a_tr], y_train,
        sample_weight=sample_w,
        validation_data=([X_q_val, X_ak_val, X_a_val], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1
    )

    y_pred_raw = model.predict([X_q_te, X_ak_te, X_a_te], verbose=0).flatten()
    y_pred     = np.clip(np.round(y_pred_raw), 1, 10)
    rmse       = np.sqrt(mean_squared_error(y_test, y_pred))
    mae        = mean_absolute_error(y_test, y_pred)
    print(f"  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}")

    fold_results.append({
        'fold'       : i + 1,
        'test_idpsj' : test_id,
        'val_idpsj'  : val_id,
        'n_train'    : len(y_train),
        'n_val'      : len(y_val),
        'n_test'     : len(y_test),
        'rmse'       : rmse,
        'mae'        : mae,
        'y_test'     : y_test,
        'y_pred'     : y_pred,
    })

    print(f"  Model saved -> {model_path}")
    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")

In [ ]:
# ── Evaluasi Akhir ────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

summary = pd.DataFrame([{
    'fold'      : r['fold'],
    'test_idpsj': r['test_idpsj'],
    'n_train'   : r['n_train'],
    'n_test'    : r['n_test'],
    'MAE'       : round(r['mae'],  4),
    'RMSE'      : round(r['rmse'], 4),
} for r in fold_results])

print('=' * 55)
print('Hasil per Fold')
print('=' * 55)
print(summary[['fold','test_idpsj','n_test','MAE','RMSE']].to_string(index=False))

print(f"{'─'*55}")
print(f"MAE  : {summary['MAE'].mean():.4f} +/- {summary['MAE'].std():.4f}")
print(f"RMSE : {summary['RMSE'].mean():.4f} +/- {summary['RMSE'].std():.4f}")
print(f"{'─'*55}")

summary.to_csv(os.path.join(OUT_DIR, 'lopo_results.csv'), index=False)

# ── Plot: MAE per fold ────────────────────────────────────────────────────────
x = np.arange(len(summary))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(x, summary['MAE'], color='steelblue', edgecolor='black')
ax.axhline(summary['MAE'].mean(), color='red', linestyle='--',
           label=f"Avg MAE {summary['MAE'].mean():.3f}")
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_title('MAE per Fold (LOPO)')
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('MAE')
ax.legend()

# ── Plot: scatter prediksi vs aktual (semua fold digabung) ───────────────────
ax2 = axes[1]
y_all_true = np.concatenate([r['y_test'] for r in fold_results])
y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])
ax2.scatter(y_all_true, y_all_pred, alpha=0.4, edgecolors='k', linewidths=0.3)
ax2.plot([1, 10], [1, 10], 'r--', label='Ideal')
ax2.set_xlabel('Grade Aktual')
ax2.set_ylabel('Grade Prediksi')
ax2.set_title('Prediksi vs Aktual (semua fold)')
ax2.set_xticks(range(1, 11))
ax2.set_yticks(range(1, 11))
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lopo_results.png'), dpi=150)
plt.show()
print("Plot disimpan -> lopo_results.png")